# Neural Data Science - Introduction to Spike Train Analysis

This notebook demonstrates basic spike train analysis techniques following the "Neural Data Science" book by Eric Lee Neu.

## Topics Covered:
- Loading and visualizing spike data
- Computing firing rates
- Creating raster plots and PSTHs
- Basic statistical analysis

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import sys

# Add Python modules to path
sys.path.append('../python')

from utils.data_utils import compute_firing_rate, smooth_firing_rate
from utils.plotting import plot_raster, plot_psth

# Set plotting style
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    plt.style.use('default')
%matplotlib inline

## 1. Generate Example Spike Data

For demonstration purposes, we'll generate synthetic spike data using a Poisson process with stimulus-related modulation.

In [ ]:
def generate_spike_data(n_trials=50, duration=2.0, base_rate=20.0):
    """Generate example spike train data."""
    np.random.seed(42)
    spike_times_list = []
    
    for trial in range(n_trials):
        # Pre-stimulus (0-0.5s): base rate
        n_pre = np.random.poisson(base_rate * 0.5)
        spikes_pre = np.sort(np.random.uniform(0, 0.5, n_pre))
        
        # Stimulus period (0.5-1.0s): elevated rate
        n_stim = np.random.poisson(base_rate * 1.5 * 0.5)
        spikes_stim = np.sort(np.random.uniform(0.5, 1.0, n_stim))
        
        # Post-stimulus (1.0-2.0s): return to base
        n_post = np.random.poisson(base_rate * 1.0)
        spikes_post = np.sort(np.random.uniform(1.0, duration, n_post))
        
        spike_times = np.concatenate([spikes_pre, spikes_stim, spikes_post])
        spike_times_list.append(spike_times)
    
    return spike_times_list

# Generate data
spike_times_list = generate_spike_data(n_trials=50)
print(f"Generated {len(spike_times_list)} trials")
print(f"Example trial 0: {len(spike_times_list[0])} spikes")

## 2. Visualize Spike Raster

A raster plot shows spike times across multiple trials.

In [ ]:
# Plot raster for first 30 trials
fig, ax = plt.subplots(figsize=(12, 6))
plot_raster(spike_times_list[:30], ax=ax, colors='black', linewidths=1)
ax.axvline(0.5, color='red', linestyle='--', alpha=0.5, label='Stimulus onset')
ax.axvline(1.0, color='blue', linestyle='--', alpha=0.5, label='Stimulus offset')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Compute and Plot PSTH

The Peri-Stimulus Time Histogram (PSTH) shows the average firing rate over time.

In [ ]:
# Concatenate all spikes and compute firing rate
all_spikes = np.concatenate(spike_times_list)
times, firing_rate = compute_firing_rate(all_spikes, bin_size=0.05, time_range=(0, 2))

# Normalize by number of trials
firing_rate = firing_rate / len(spike_times_list)

# Smooth the firing rate
firing_rate_smooth = smooth_firing_rate(firing_rate, window_size=5)

# Plot
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(times, firing_rate, alpha=0.4, label='Raw', color='gray')
ax.plot(times, firing_rate_smooth, linewidth=2, label='Smoothed', color='blue')
ax.axvline(0.5, color='red', linestyle='--', alpha=0.5, label='Stimulus onset')
ax.axvline(1.0, color='red', linestyle='--', alpha=0.5)
ax.axhline(20, color='green', linestyle=':', alpha=0.5, label='Base rate')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Firing Rate (Hz)')
ax.set_title('Peri-Stimulus Time Histogram (PSTH)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Statistical Analysis

Compute basic statistics about the spike trains.

In [ ]:
# Compute spike counts per trial
spike_counts = np.array([len(st) for st in spike_times_list])

print("Spike Train Statistics:")
print("=" * 50)
print(f"Number of trials: {len(spike_times_list)}")
print(f"Mean spikes per trial: {spike_counts.mean():.2f} ± {spike_counts.std():.2f}")
print(f"Min spikes per trial: {spike_counts.min()}")
print(f"Max spikes per trial: {spike_counts.max()}")
print(f"\nPeak firing rate: {firing_rate_smooth.max():.2f} Hz")
print(f"Mean baseline rate (0-0.5s): {firing_rate_smooth[:10].mean():.2f} Hz")
print(f"Mean stimulus rate (0.5-1.0s): {firing_rate_smooth[10:20].mean():.2f} Hz")

# Plot spike count distribution
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(spike_counts, bins=15, edgecolor='black', alpha=0.7)
ax.axvline(spike_counts.mean(), color='red', linestyle='--', linewidth=2, label='Mean')
ax.set_xlabel('Spike Count per Trial')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Spike Counts')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Next Steps

This notebook covered the basics of spike train analysis. To continue learning:

1. **Load real data**: Replace synthetic data with actual neural recordings
2. **Advanced analysis**: Explore inter-spike intervals, Fano factors, cross-correlations
3. **LFP analysis**: Analyze local field potentials and their relationship to spiking
4. **Decoding**: Build models to decode stimuli from neural activity

See the Neural Data Science book for more advanced topics!